In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://bh.rdm.yzwlab.com/'
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

tljh_url = 'http://localhost'
tljh_username = 'admin'
tljh_password = 'change-your-password'

project_name = None

binderhub_base_image = 'yacchin1205/cs-binder-dockerfiles:python-test-lab-4.2.6'
binderhub_cran_package = 'eegkit'
binderhub_rgithub_package = 'r-lib/roxygen2'
binderhub_post_build_script = '''#!/bin/bash
date > ~/uptime'''
binderhub_dockerfile_option = '!SINGLESPEED!'
binderhub_launch_timeout = 1800000  # 30 minutes
dockerfile_custom_script = """FROM niicloudoperation/notebook:feature-lab

COPY --chown=$NB_UID:$NB_GID . .
"""
binderhub_test_filename = 'grdm_test_file.txt'
binderhub_test_file_content = 'GRDM_FILE_SYNC_TEST'


In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if tljh_username is None:
    tljh_username = input('TLJH username')
if tljh_password is None:
    tljh_password = getpass(prompt=f'Password for {tljh_username}@TLJH')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

binderhub_url = tljh_url
project_url = None
project_created = False
environment_name = None


# BinderHubアドオン Dockerfile による解析環境の起動

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アドオン操作
- シナリオ名: Dockerfile基本イメージを使った解析環境の起動
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること / JupyterHubはサーバーが5つ以内の状態であること)、BinderHub OAuthクライアント情報
- 事前条件: 「BinderHubアドオン repo2docker による解析環境の起動」を実施済みであること

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


In [ ]:
import asyncio

async def run_lab_command(page, command, expected_substring):
    editor = page.locator('div.jp-CodeCell div.cm-content').nth(0)
    await expect(editor).to_be_visible(timeout=transition_timeout)
    await editor.click()
    await editor.fill(command)

    run_button = page.locator('//jp-button[@data-command = "notebook:run-cell-and-select-next"]')
    await expect(run_button).to_be_visible(timeout=transition_timeout)
    await run_button.click()

    output = page.locator(f'//*[contains(@class, "jp-OutputArea")]//*[text()[contains(., "{expected_substring}")]]')
    await expect(output).to_be_visible(timeout=transition_timeout)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await expect(page.locator('//button[text() = "ログイン"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 基本イメージの「変更」をクリックする

基本イメージ一覧が表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-image-change]').click()
    await expect(page.locator(f'//*[@data-test-image-selection = "{binderhub_base_image}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Data Science Notebook」をクリックする

「追加パッケージ」エディターが表示され、既定パッケージが確認できること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-image-selection = "{binderhub_base_image}"]').click()
    await expect(page.locator('//*[@data-test-package-add]')).not_to_have_count(0, timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "apt"]//*[text() = "sl"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "conda"]//*[text() = "awscli"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-test-package-editor = "pip"]//*[text() = "papermill"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「R (CRAN)」の「+追加」をクリックする

パッケージ名の入力欄が表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "rcran"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_cran_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "rcran"]//*[text() = "{binderhub_cran_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(10)

await run_pw(_step)

## 「R (GitHub)」の「+追加」をクリックする

パッケージ名の入力欄が表示されること

In [ ]:
async def _step(page):
    await page.locator('//div[@data-test-package-editor = "rgithub"]//*[@data-test-package-add]').click()
    field = page.locator('//input[@name = "package_name"]')
    await expect(field).to_be_visible(timeout=transition_timeout)
    await field.fill(binderhub_rgithub_package)
    await page.locator('//button[@data-test-package-item-confirm]').click()
    await expect(page.locator(f'//div[@data-test-package-editor = "rgithub"]//*[text() = "{binderhub_rgithub_package}"]')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(10)

await run_pw(_step)

## 「自動実行スクリプト」をクリックする

スクリプトエディタが既定値を保持していること

In [ ]:
async def _step(page):
    await page.locator('//h3//button/*[contains(@class, "fa-chevron-right")]').click()
    textarea = page.locator('//textarea[@data-test-post-build]')
    await expect(textarea).to_be_visible(timeout=transition_timeout)
    await expect(textarea).to_have_value(binderhub_post_build_script, timeout=transition_timeout)
    save_button = page.locator('//button[@data-test-save-post-build]')
    await expect(save_button).to_be_visible(timeout=transition_timeout)
    await expect(save_button).to_be_disabled(timeout=transition_timeout)
    await page.wait_for_load_state('networkidle')

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「ファイル」をクリックする

フォルダ「.binder」が存在していること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-node-navbar-link = "files"]').click()
    binder_folder = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await expect(binder_folder).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「.binder」フォルダを開く

ファイル「Dockerfile」「postBuild」が作成されていること

In [ ]:
async def _step(page):
    folder_toggle = page.locator('//*[text() = ".binder"]/../..//*[contains(@class, "fa-plus")]')
    await folder_toggle.click()
    for filename in ['Dockerfile', 'postBuild']:
        await expect(page.locator(f'//*[text() = "{filename}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    await expect(page.locator('//*[@data-test-binderhub-launch]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

TLJHのログインページが表示されること

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=transition_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    await expect(popup.locator('#username_input')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)


## TLJHのユーザー名・パスワードを入力し、Sign inをクリックする

TLJHのダッシュボードに遷移すること

In [ ]:
environment_name = None

async def _step(page):
    global environment_name
    await page.locator('#username_input').fill(tljh_username)
    await page.locator('#password_input').fill(tljh_password)
    await page.locator('#login_submit').click()

    sync_icon = page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="SyncIcon"]')
    await expect(sync_icon).to_be_visible(timeout=transition_timeout)
    await sync_icon.click()

    env_locator = page.locator('//*[@data-testid="SyncIcon"]/../../..//*[@data-field="display_name"]//*[@role="presentation"]')
    environment_name = await env_locator.text_content()
    print(environment_name)
    await asyncio.sleep(3)

await run_pw(_step)

## Statusがチェックマークになるまで待つ


In [ ]:
import traceback

async def _step(page):
    global environment_name
    await page.locator('//*[contains(text(), "Close")]').click()
    retry_count = 40
    while retry_count > 0:
        try:
            check_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="CheckIcon"]')
            await expect(check_icon).to_be_visible(timeout=transition_timeout)
            break
        except:
            retry_count -= 1
            if retry_count == 0:
                traceback.print_exc()
            await page.reload()

    target_check_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="CheckIcon"]')
    try:
        await expect(target_check_icon).to_be_visible(timeout=transition_timeout)
    except:
        try:
            sync_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="SyncIcon"]')
            await sync_icon.click()
            await asyncio.sleep(30)
        except:
            traceback.print_exc()
        raise

await run_pw(_step)

## 「Servers」をクリックする

サーバー一覧画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href="/services/tljh_repo2docker/servers"]').click()
    await expect(page.locator('//button[contains(text(), "Create new Server")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Create new Server」をクリックする

サーバー作成ダイアログが表示されること

In [ ]:
async def _step(page):
    global environment_name
    await page.locator('//button[contains(text(), "Create new Server")]').click()
    checkbox = page.locator(f'//*[@title="{environment_name}"]/../..//input[@type="checkbox"]')
    await expect(checkbox).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## このプロジェクトURLに対応する環境のチェックボックスをクリックし、サーバー名に「test-server」と入力し、「Create Server」をクリックする

In [ ]:
async def _step(page):
    await page.locator(f'//*[@title="{environment_name}"]/../..//input[@type="checkbox"]').click()
    await page.locator('#server_name').fill('test-server')
    await page.locator('//button[contains(text(), "Create Server")]').click()
    await expect(page.locator('#server_name')).not_to_be_visible(timeout=transition_timeout * 5)

await run_pw(_step)

## 「Open Server」をクリックする

JupyterLabウィンドウが表示されること

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup')
    await page.locator('//a[contains(text(), "Open Server")]').click()
    popup = await popup_future
    await expect(popup.locator('//*[@class="jp-LauncherCard" and @title="Python 3 (ipykernel)" and @data-category="Notebook"]')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)


## 新規Notebookを作成する

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「!which /usr/games/sl」を実行して結果を確認する

コードセルの下に `/usr/games/sl` が表示されること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!which /usr/games/sl', '/usr/games/sl')

await run_pw(_step)


## 「!aws」を実行して結果を確認する

コードセルの下に `usage: aws` を含む文字列が表示されること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!aws', 'usage: aws')

await run_pw(_step)


## 「!papermill」を実行して結果を確認する

コードセルの下に `Usage: papermill` を含む文字列が表示されること

In [ ]:
async def _step(page):
    await run_lab_command(page, '!papermill', 'Usage: papermill')

await run_pw(_step)


## GRDMプロジェクトのファイルがコピーされていることを確認する

テストファイルの内容が出力されること

In [ ]:
async def _step(page):
    await run_lab_command(page, f'!cat {binderhub_test_filename}', binderhub_test_file_content)

await run_pw(_step)


## Notebookを閉じる

In [ ]:
async def _step(page):
    save_button = page.locator('//jp-button[@data-command = "docmanager:save"]')
    await expect(save_button).to_be_visible(timeout=transition_timeout)
    await save_button.click()

    rename_dialog = page.locator('//div[text() = "Rename"]')
    await expect(rename_dialog).to_be_visible(timeout=transition_timeout)
    await rename_dialog.click()
    await expect(rename_dialog).not_to_be_visible(timeout=transition_timeout)

    close_icon = page.locator('//*[contains(@class, "lm-mod-current")]//*[contains(@class, "lm-TabBar-tabCloseIcon")]')
    await expect(close_icon).to_be_visible(timeout=transition_timeout)
    await close_icon.click()

    launcher_label = 'R'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## JupyterLabウィンドウを閉じる

TLJHのServersページに戻ること

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## 「Stop Server」をクリックし、「Accept」をクリックする

サーバーが停止すること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(text(), "Stop Server")]').click()
    await page.locator('//*[contains(text(), "Accept")]').click()
    await expect(page.locator('//a[contains(text(), "Open Server")]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## TLJHのブラウザタブを閉じ、GRDMの解析ページに戻る

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## 「デフォルトストレージの内容をコピーする」チェックボックスをオフにする

チェックボックスがオフになること

In [ ]:
async def _step(page):
    checkbox = page.locator('//*[@data-test-copy-default-storage]')
    await expect(checkbox).to_be_checked(timeout=transition_timeout)
    await checkbox.click()

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

TLJHのダッシュボードに遷移すること

In [ ]:
nocopy_environment_name = None

async def _step(page):
    global nocopy_environment_name
    popup_future = page.wait_for_event('popup', timeout=transition_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future

    sync_icon = popup.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="SyncIcon"]')
    await expect(sync_icon).to_be_visible(timeout=transition_timeout)
    await sync_icon.click()

    env_locator = popup.locator('//*[@data-testid="SyncIcon"]/../../..//*[@data-field="display_name"]//*[@role="presentation"]')
    nocopy_environment_name = await env_locator.text_content()
    print(nocopy_environment_name)
    await asyncio.sleep(3)

    return popup

await run_pw(_step)


## Statusがチェックマークになるまで待つ

In [ ]:
async def _step(page):
    await page.locator('//*[contains(text(), "Close")]').click()
    retry_count = 40
    while retry_count > 0:
        try:
            check_icon = page.locator(f'//*[@title="{nocopy_environment_name}"]/../..//*[@data-testid="CheckIcon"]')
            await expect(check_icon).to_be_visible(timeout=transition_timeout)
            break
        except:
            retry_count -= 1
            if retry_count == 0:
                traceback.print_exc()
            await page.reload()

await run_pw(_step)

## サーバーを作成し、JupyterLabを起動する

In [ ]:
async def _step(page):
    await page.locator('//a[@href="/services/tljh_repo2docker/servers"]').click()
    await expect(page.locator('//button[contains(text(), "Create new Server")]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(text(), "Create new Server")]').click()
    checkbox = page.locator(f'//*[@title="{nocopy_environment_name}"]/../..//input[@type="checkbox"]')
    await expect(checkbox).to_be_visible(timeout=transition_timeout)
    await checkbox.click()
    await page.locator('#server_name').fill('nocopy-server')
    await page.locator('//button[contains(text(), "Create Server")]').click()
    await expect(page.locator('#server_name')).not_to_be_visible(timeout=transition_timeout * 5)
    popup_future = page.wait_for_event('popup')
    await page.locator('//a[contains(text(), "Open Server")]').click()
    popup = await popup_future
    await expect(popup.locator('//*[@class="jp-LauncherCard" and @title="Python 3 (ipykernel)" and @data-category="Notebook"]')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)


## 新規Notebookを作成する

In [ ]:
async def _step(page):
    launcher_label = 'Python 3 (ipykernel)'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## GRDMプロジェクトのファイルがコピーされていないことを確認する

テストファイルが存在せず、エラーが出力されること

In [ ]:
async def _step(page):
    await run_lab_command(page, f'!cat {binderhub_test_filename}', 'No such file or directory')

await run_pw(_step)


## JupyterLabウィンドウを閉じる

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## 「Stop Server」をクリックし、「Accept」をクリックする

In [ ]:
async def _step(page):
    await page.locator('//*[contains(text(), "Stop Server")]').click()
    await page.locator('//*[contains(text(), "Accept")]').click()
    await expect(page.locator('//a[contains(text(), "Open Server")]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## TLJHのブラウザタブを閉じ、GRDMの解析ページに戻る

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## 「デフォルトストレージの内容をコピーする」チェックボックスをオンに戻す

チェックボックスがオンになること

In [ ]:
async def _step(page):
    checkbox = page.locator('//*[@data-test-copy-default-storage]')
    await expect(checkbox).not_to_be_checked(timeout=transition_timeout)
    await checkbox.click()

await run_pw(_step)


## 基本イメージの「変更」をクリックする

Dockerfileオプションが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-test-image-change]').click()
    await expect(page.locator(f'//*[@data-test-image-selection = "{binderhub_dockerfile_option}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Dockerfileを用いたカスタムイメージ」をクリックする

「Dockerfile」エディターが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-image-selection = "{binderhub_dockerfile_option}"]').click()
    await expect(page.locator('//h3[text() = "Dockerfile"]/../..//textarea')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「Dockerfile」エディターにスクリプトを入力する

入力内容が表示されること

In [ ]:
async def _step(page):
    textarea = page.locator('//h3[text() = "Dockerfile"]/../..//textarea')
    await textarea.fill(dockerfile_custom_script)
    await textarea.press('Tab')

await run_pw(_step)


## 「保存」をクリックする

エラーメッセージが表示されないこと

In [ ]:
async def _step(page):
    await page.locator('//h3[text() = "Dockerfile"]/../..//button[text() = "保存"]').click()
    await asyncio.sleep(10)

await run_pw(_step)


## 「新しい解析環境を作成」をクリックする

TLJHのログインページが表示されること

In [ ]:
async def _step(page):
    popup_future = page.wait_for_event('popup', timeout=transition_timeout)
    await page.locator('//*[@data-test-binderhub-launch]').click()
    popup = await popup_future
    await expect(popup.locator('//*[@data-testid="SyncIcon"]')).to_be_visible(timeout=transition_timeout)
    return popup

await run_pw(_step)

## Statusがチェックマークになるまで待つ

In [ ]:
import traceback

async def _step(page):
    global environment_name

    sync_icon = page.locator(f'//a[text()="{project_url}"]/../..//*[@data-testid="SyncIcon"]')
    await expect(sync_icon).to_be_visible(timeout=transition_timeout)
    await sync_icon.click()
    await asyncio.sleep(3)

    env_locator = page.locator('//*[@data-testid="SyncIcon"]/../../..//*[@data-field="display_name"]//*[@role="presentation"]')
    environment_name = await env_locator.text_content()
    print(environment_name)

    await page.locator('//*[contains(text(), "Close")]').click()

    retry_count = 40
    while retry_count > 0:
        try:
            check_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="CheckIcon"]')
            await expect(check_icon).to_be_visible(timeout=transition_timeout)
            break
        except:
            retry_count -= 1
            if retry_count == 0:
                traceback.print_exc()
            await page.reload()

    target_check_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="CheckIcon"]')
    try:
        await expect(target_check_icon).to_be_visible(timeout=transition_timeout)
    except:
        try:
            sync_icon = page.locator(f'//*[@title="{environment_name}"]/../..//*[@data-testid="SyncIcon"]')
            await sync_icon.click()
            await asyncio.sleep(30)
        except:
            traceback.print_exc()
        raise

await run_pw(_step)

## サーバーを作成し、JupyterLabを起動する

In [ ]:
async def _step(page):
    global environment_name
    await page.locator('//a[@href="/services/tljh_repo2docker/servers"]').click()
    await expect(page.locator('//button[contains(text(), "Create new Server")]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//button[contains(text(), "Create new Server")]').click()
    checkbox = page.locator(f'//*[@title="{environment_name}"]/../..//input[@type="checkbox"]')
    await expect(checkbox).to_be_visible(timeout=transition_timeout)
    await checkbox.click()
    await page.locator('#server_name').fill('dockerfile-server')
    await page.locator('//button[contains(text(), "Create Server")]').click()
    await expect(page.locator('#server_name')).not_to_be_visible(timeout=transition_timeout * 5)
    popup_future = page.wait_for_event('popup')
    await page.locator('//a[contains(text(), "Open Server")]').click()
    popup = await popup_future
    await expect(popup.locator('//h1[@data-jupyter-id="Jupyter-Notebook-for-Literate-Computing-for-Reproducible-Infrastructure"]')).to_be_visible(timeout=transition_timeout * 3)
    return popup

await run_pw(_step)

## GRDMプロジェクトのファイルがコピーされていることを確認する

新規Notebookを作成し、テストファイルの内容が出力されること

In [ ]:
async def _step(page):
    await page.locator('//*[@data-id="launcher-0"]').click()
    launcher_label = 'Python 3'
    launcher = page.locator(f'//*[@class = "jp-LauncherCard" and @title = "{launcher_label}" and @data-category = "Notebook"]')
    await expect(launcher).to_be_visible(timeout=transition_timeout)
    await launcher.click()

    # Execution Indicator は、 ... でCollapseされ表示されない可能性がある
    try:
        await expect(page.locator('//*[contains(@class, "jp-Notebook-ExecutionIndicator") and @data-status="idle"]')).to_be_visible(timeout=transition_timeout)

        await asyncio.sleep(5)
    except:
        # Warning
        traceback.print_exc()

    await run_lab_command(page, f'!cat {binderhub_test_filename}', binderhub_test_file_content)

await run_pw(_step)

## JupyterLabウィンドウを閉じる

TLJHのServersページに戻ること

In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


## サーバーを停止し、TLJHタブを閉じる

In [ ]:
async def _step(page):
    await page.locator('//*[contains(text(), "Stop Server")]').click()
    await page.locator('//*[contains(text(), "Accept")]').click()
    await expect(page.locator('//a[contains(text(), "Open Server")]')).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)


In [ ]:
await close_latest_page()

async def _step(page):
    pass

await run_pw(_step)


終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}